# CE541E08 — Unit 4 · Day 36 — Complete River Basin Analysis — Unit 4 Capstone
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 36 of 45 |
| **Topics** | dataset overview · annual stats · climatology · seasonal · decade trend |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 36"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — The Complete River Basin Analysis Workflow

Day 36 is the capstone for Unit 4. All four code blocks build one complete hydrological analysis of a 20-year river basin dataset — the kind of analysis a junior hydrologist would produce for a dam safety review or water resources planning study.

**Part A — Dataset Overview:** Load, inspect, and summarise the 20-year dataset.

**Part B — Annual Statistics:** Compute annual mean, peak, and total for each year. Compare first and second decade.

**Part C — Monthly Climatology:** Long-term average and seasonal pattern. Compare monsoon vs non-monsoon.

All code uses what was covered in Days 28–35. No new functions are introduced — this session is about combining them into a coherent workflow.

---
## Code Block 1 — Part A: Dataset Overview

### What this code does

We build a 20-year × 12-month DataFrame (240 records) with both streamflow and rainfall. We use `.describe()` to get a statistical overview and extract the year and month from the Period string column.

### Why each step is taken

**String slicing `df['Period'].str[:4].astype(int)`:**
The Period column contains strings like '2004-Jan'. `.str[:4]` extracts the first 4 characters (the year) as strings. `.astype(int)` converts to integer. `.str[5:]` extracts 'Jan', 'Feb', etc.

**`df.describe()`:**
One call gives count, mean, std, min, Q25, median, Q75, max for both numeric columns simultaneously.

### Algorithm

```
1. Build 240-row DataFrame: Period, Flow_m3s, Rainfall_mm

2. Extract Year from Period string: .str[:4].astype(int)
   Extract Month from Period string: .str[5:]

3. df.describe() → statistical overview of both columns
```

### Expected output

```
==================================================
PART A — DATASET OVERVIEW
--------------------------------------------------
Records : 240
Years   : 2004 - 2023

       Flow_m3s  Rainfall_mm
count     240.0        240.0
mean      ...          ...
...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(42)
years      = list(range(2004, 2024))
months     = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates_list = [f"{y}-{m}" for y in years for m in months]
base       = np.array([45, 38, 28, 22, 35, 234, 456, 389, 198, 89, 62, 50])
rain_base  = np.array([8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13])
flows = np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.25,240),5),1)
rain  = np.round(np.maximum(np.tile(rain_base,20)+np.random.normal(0,rain_base*0.2,240),0),1)
df = pd.DataFrame({'Period':dates_list,'Flow_m3s':flows,'Rainfall_mm':rain})

# Extract year and month from the Period string column
df['Year']  = df['Period'].str[:4].astype(int)
df['Month'] = df['Period'].str[5:]

print("=" * 50)
print("PART A — DATASET OVERVIEW")
print("-" * 50)
print(f"Records : {len(df)}")
print(f"Years   : {df['Year'].min()} - {df['Year'].max()}")
print()
print(df.describe().round(1))

### 🔁 Try this

After running `.describe()`, answer these questions from the output alone:

- What fraction of months have flow above the 75th percentile?
- Is the flow distribution symmetric? (Compare mean vs median = 50%)

---
## Code Block 2 — Part B: Annual Statistics

### What this code does

We group by Year to compute annual mean, peak, and total flow, then compare the first decade (2004-2013) to the second (2014-2023).

### Why each step is taken

**`.agg({'Flow_m3s':['mean','max'],'Rainfall_mm':'sum'})`:**
`agg` with a dictionary applies different functions to different columns. This computes mean and max for flow, but only the total (sum) for rainfall — both in a single `groupby` call.

**`annual.columns = [...]`:**
Renaming multi-level columns after `agg` with mixed functions to simple readable names.

**`annual.iloc[:10]` vs `annual.iloc[10:]`:**
Selecting first 10 rows (2004-2013) and last 10 rows (2014-2023) by position. `.mean()` on the sliced rows gives the decade average.

### Algorithm

```
1. df.groupby('Year').agg({'Flow_m3s':['mean','max'],'Rainfall_mm':'sum'})
   → one row per year, three columns

2. Rename columns: Mean_Flow, Peak_Flow, Annual_Rain

3. annual.iloc[:10]['Mean_Flow'].mean() → first decade
   annual.iloc[10:]['Mean_Flow'].mean() → second decade

4. Compare: (recent-early)/early*100 → % change
```

### Expected output

```
======================================================
PART B — ANNUAL STATISTICS
------------------------------------------------------
      Mean_Flow  Peak_Flow  Annual_Rain
Year
2004       ...       ...         ...
...
2023       ...       ...         ...

First decade mean flow  : ... m3/s
Second decade mean flow : ... m3/s
Change                  : +/-...%
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(42)
years=list(range(2004,2024)); months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates_list=[f"{y}-{m}" for y in years for m in months]
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
rain_base=np.array([8,12,18,52,87,134,118,113,95,71,44,13])
flows=np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.25,240),5),1)
rain=np.round(np.maximum(np.tile(rain_base,20)+np.random.normal(0,rain_base*0.2,240),0),1)
df=pd.DataFrame({'Period':dates_list,'Flow_m3s':flows,'Rainfall_mm':rain})
df['Year']=df['Period'].str[:4].astype(int)

# Multi-column agg: different functions for different columns
annual = df.groupby('Year').agg({'Flow_m3s':['mean','max'],'Rainfall_mm':'sum'})
annual.columns = ['Mean_Flow','Peak_Flow','Annual_Rain']

print("=" * 55)
print("PART B — ANNUAL STATISTICS")
print("-" * 55)
print(annual.round(1))
print()

# Decade comparison: first 10 rows vs last 10 rows of the annual summary
early  = annual.iloc[:10]['Mean_Flow'].mean()
recent = annual.iloc[10:]['Mean_Flow'].mean()
print(f"First decade mean flow  : {early:.1f} m3/s")
print(f"Second decade mean flow : {recent:.1f} m3/s")
print(f"Change                  : {(recent-early)/early*100:+.1f}%")

### 🔁 Try this

Find the **wettest year by rainfall** and the **year with the highest peak flow** — do they coincide?

Use `annual['Annual_Rain'].idxmax()` and `annual['Peak_Flow'].idxmax()`

---
## Code Block 3 — Part C: Monthly Climatology and Seasonal Analysis

### What this code does

We compute the 20-year monthly climatology (mean flow and rainfall for each calendar month) and compare monsoon vs non-monsoon seasonal means. We also plot dual bar charts.

### Why each step is taken

**`df.groupby('Month_num')[['Flow_m3s','Rainfall_mm']].mean()`:**
Groups by month number (1=Jan, ..., 12=Dec) across all 20 years. For each month, computes the mean of both flow and rainfall simultaneously. The result is a `(12,2)` climatology table.

**`df['Season'].apply(lambda m: ...)`:**
Creates a season label for each row based on the Month_num column. Used for the seasonal groupby that follows.

**Dual subplot `fig,axes=plt.subplots(1,2,...)`:**
Creates two side-by-side bar charts — one for flow, one for rainfall. Both share the same month x-axis, making comparison straightforward.

### Algorithm

```
1. df['Month_num'] = month index (1-12) from Period string

2. df.groupby('Month_num')[['Flow_m3s','Rainfall_mm']].mean()
   → (12,2) climatology: mean per calendar month

3. df['Season'] = apply(lambda: 'Monsoon' if 6<=m<=9 else 'Non-monsoon')

4. df.groupby('Season')[['Flow_m3s','Rainfall_mm']].mean()
   → 2-row seasonal summary

5. Plot dual bar charts: flow (left) and rainfall (right)
```

### Expected output

```
==================================================
PART C — MONTHLY CLIMATOLOGY
--------------------------------------------------
     Flow_m3s  Rainfall_mm
Jan      ...         ...
...
Seasonal means:
                 Flow_m3s  Rainfall_mm
Monsoon(Jun-Sep)    ...        ...
Non-monsoon         ...        ...
```

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
months_list=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
rain_base=np.array([8,12,18,52,87,134,118,113,95,71,44,13])
rows=[]
for yr in range(2004,2024):
    for i,m in enumerate(months_list):
        rows.append({'Year':yr,'Month':m,'Month_num':i+1,
                     'Flow_m3s':round(base[i]+np.random.normal(0,base[i]*0.25),1),
                     'Rainfall_mm':round(rain_base[i]+np.random.normal(0,rain_base[i]*0.2),1)})
df = pd.DataFrame(rows)
df['Season'] = df['Month_num'].apply(
    lambda m: 'Monsoon(Jun-Sep)' if 6<=m<=9 else 'Non-monsoon'
)

# 20-year monthly climatology
clim = df.groupby('Month_num')[['Flow_m3s','Rainfall_mm']].mean()
clim.index = months_list

print("=" * 50)
print("PART C — MONTHLY CLIMATOLOGY")
print("-" * 50)
print(clim.round(1))

# Dual bar charts
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].bar(range(1,13), clim['Flow_m3s'], color='steelblue')
axes[0].set_xticks(range(1,13)); axes[0].set_xticklabels(months_list,rotation=45)
axes[0].set_title('Mean Monthly Flow'); axes[0].set_ylabel('m3/s')
axes[1].bar(range(1,13), clim['Rainfall_mm'], color='green')
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(months_list,rotation=45)
axes[1].set_title('Mean Monthly Rainfall'); axes[1].set_ylabel('mm')
plt.tight_layout(); plt.show()

season = df.groupby('Season')[['Flow_m3s','Rainfall_mm']].mean()
print("Seasonal means:"); print(season.round(1))

### 🔁 Try this

Compute the **monsoon contribution** to the annual total:

- `monsoon_flow = clim.loc[['Jun','Jul','Aug','Sep'],'Flow_m3s'].sum()`
- `annual_flow  = clim['Flow_m3s'].sum()`
- `pct = monsoon_flow / annual_flow * 100`

What fraction of annual flow occurs in the monsoon months?

---
## Unit 4 Summary — All Pandas Concepts

| Day | Key concept | Core function/method |
|---|---|---|
| 28 | Series, DataFrame, read_csv | `pd.Series`, `pd.DataFrame`, `pd.read_csv` |
| 29 | Inspect and filter | `.info()`, `.describe()`, `df[condition]`, `.nlargest()` |
| 30 | loc, iloc, DatetimeIndex | `.loc['label']`, `.iloc[n]`, `df.loc['2024-07']` |
| 31 | Merge and concat | `pd.merge(how='inner/outer/left')`, `pd.concat` |
| 32 | groupby and resample | `.groupby('Year').agg(...)`, `.resample('ME')` |
| 33 | apply and map | `.apply(func, axis=1)`, `.map(dict)` |
| 34 | pd.cut and replace | `pd.cut(bins, labels)`, `.replace({})`, `pd.crosstab` |
| 35 | Visualisation | line, scatter, boxplot, bar+trend |
| 36 | Complete analysis | All combined — dataset to insights |

---
## Day 36 — Unit 4 Capstone Assignment

Submit a complete river basin analysis notebook using the 20-year dataset from this session.

**Required outputs:**
1. Dataset overview: shape, date range, describe()
2. Annual statistics table: mean, peak, total flow per year
3. Decade comparison with % change
4. Monthly climatology table and dual bar chart
5. Seasonal means (Monsoon vs Non-monsoon)
6. One-paragraph interpretation: what does the trend tell a water resources engineer?

**Submit to GitHub with commit message:** `Day 36 — Unit 4 capstone completed`

### ▶ Capstone cell

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
years=list(range(2004,2024)); months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dates_list=[f"{y}-{m}" for y in years for m in months]
base=np.array([45,38,28,22,35,234,456,389,198,89,62,50])
rain_base=np.array([8,12,18,52,87,134,118,113,95,71,44,13])
flows=np.round(np.maximum(np.tile(base,20)+np.random.normal(0,base*0.25,240),5),1)
rain=np.round(np.maximum(np.tile(rain_base,20)+np.random.normal(0,rain_base*0.2,240),0),1)
df=pd.DataFrame({'Period':dates_list,'Flow_m3s':flows,'Rainfall_mm':rain})
df['Year']=df['Period'].str[:4].astype(int)
df['Month_num']=df['Period'].apply(lambda p:months.index(p[5:])+1)

# 1. Overview
print(f"Records: {len(df)}  Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.describe().round(1))

# 2. Annual statistics
annual=df.groupby('Year').agg({'Flow_m3s':['mean','max'],'Rainfall_mm':'sum'})
annual.columns=['Mean_Flow','Peak_Flow','Annual_Rain']
print(annual.round(1))

# 3. Decade comparison
early=annual.iloc[:10]['Mean_Flow'].mean()
recent=annual.iloc[10:]['Mean_Flow'].mean()
print(f"First decade: {early:.1f}  Second decade: {recent:.1f}  Change: {(recent-early)/early*100:+.1f}%")

# 4-5. Climatology and seasonal
clim=df.groupby('Month_num')[['Flow_m3s','Rainfall_mm']].mean()
clim.index=months
print(clim.round(1))

df['Season']=df['Month_num'].apply(lambda m:'Monsoon(Jun-Sep)' if 6<=m<=9 else 'Non-monsoon')
print(df.groupby('Season')[['Flow_m3s','Rainfall_mm']].mean().round(1))

# 6. Your interpretation:
print("\nInterpretation:")
print("The 20-year record shows ...")

---
- [ ] Run all cells — verify each Part outputs the expected results
- [ ] Complete the interpretation paragraph in the capstone cell
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day36.ipynb`
- [ ] Commit message: `Day 36 — Unit 4 capstone completed`

**Unit 5 starts next: Introduction to MATLAB**

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*